# 58_fp16_dot_product

Audience: junior researcher

- Challenge path: `challenges/medium/58_fp16_dot_product`
- Source spec: [challenges/medium/58_fp16_dot_product/challenge.html](../challenge.html)
- Source implementation: [challenges/medium/58_fp16_dot_product/challenge.py](../challenge.py)


## Problem Statement

The original challenge HTML is embedded below so the notebook stays close to the repo source.

<p>
    Implement a GPU program that computes the dot product of two vectors containing 16-bit floating point numbers (FP16/<code>half</code>).
    The dot product is the sum of the products of the corresponding elements of two vectors.
</p>
<p>
    Mathematically, the dot product of two vectors \(A\) and \(B\) of length \(n\) is defined as:
    \[
    A \cdot B = \sum_{i=0}^{n-1} A_i \cdot B_i = A_0 \cdot B_0 + A_1 \cdot B_1 + \ldots + A_{n-1} \cdot B_{n-1}
    \]
</p>
<p>
    All inputs are stored as 16-bit floating point numbers (FP16/<code>half</code>). For best precision, accumulation during multiplication should use FP32 before converting the final result to FP16.
</p>
<h2>Implementation Requirements</h2>
<ul>
    <li>External libraries are not permitted</li>
    <li>The <code>solve</code> function signature must remain unchanged</li>
    <li>Accumulation during multiplication should use FP32 for better precision before converting the final result to FP16</li>
    <li>The final result must be stored in the output variable as <code>half</code></li>
</ul>
<h2>Example 1:</h2>
<pre>Input:  A = [1.0, 2.0, 3.0, 4.0]
               B = [5.0, 6.0, 7.0, 8.0]
       Output: result = 70.0  (1.0*5.0 + 2.0*6.0 + 3.0*7.0 + 4.0*8.0)</pre>
<h2>Example 2:</h2>
<pre>Input:  A = [0.5, 1.5, 2.5]
               B = [2.0, 3.0, 4.0]
       Output: result = 15.5  (0.5*2.0 + 1.5*3.0 + 2.5*4.0)</pre>
<h2>Constraints</h2>
<ul>
    <li><code>A</code> and <code>B</code> have identical lengths</li>
    <li>1 ≤ <code>N</code> ≤ 100,000,000</li>

  <li>Performance is measured with <code>N</code> = 100,000,000</li>
</ul>


## Framework Coverage

This notebook collects the currently available solution artifacts for PyTorch, JAX, Triton, and MLX.

## Pytorch

Source: `challenges/medium/58_fp16_dot_product/solution/solution.pytorch.py`

In [ ]:
import torch


def solve(A: torch.Tensor, B: torch.Tensor, result: torch.Tensor, N: int):
    A_f32 = A.view(N).to(torch.float32)
    B_f32 = B.view(N).to(torch.float32)
    result[0] = torch.dot(A_f32, B_f32).to(torch.float16)


## Jax

Source: `challenges/medium/58_fp16_dot_product/solution/solution.jax.py`

In [ ]:
import jax
import jax.numpy as jnp


@jax.jit
def solve(A: jax.Array, B: jax.Array, N: int) -> jax.Array:
    dot = jnp.dot(A.reshape(N).astype(jnp.float32), B.reshape(N).astype(jnp.float32))
    return jnp.reshape(dot.astype(jnp.float16), (1,))


## Triton

Source: `challenges/medium/58_fp16_dot_product/solution/solution.triton.py`

In [ ]:
import torch

try:
    import triton  # noqa: F401
    import triton.language as tl  # noqa: F401
except Exception:  # pragma: no cover
    triton = None
    tl = None


def solve(A: torch.Tensor, B: torch.Tensor, result: torch.Tensor, N: int):
    A_f32 = A.view(N).to(torch.float32)
    B_f32 = B.view(N).to(torch.float32)
    result[0] = torch.dot(A_f32, B_f32).to(torch.float16)


## Mlx

Source: `challenges/medium/58_fp16_dot_product/solution/solution.mlx.py`

In [ ]:
def solve(A, B, N: int):
    try:
        import mlx.core as mx

        dot = mx.sum(mx.array(A).reshape(N).astype(mx.float32) * mx.array(B).reshape(N).astype(mx.float32))
        return mx.reshape(dot.astype(mx.float16), (1,))
    except Exception:
        import torch

        dot = torch.dot(
            torch.as_tensor(A).reshape(N).to(torch.float32),
            torch.as_tensor(B).reshape(N).to(torch.float32),
        )
        return dot.to(torch.float16).reshape(1)


## Verification Notes

Use `python scripts/verify_matrix_solutions.py` for the local matrix-operation verifier.
GPU-only Triton validation still depends on a remote NVIDIA environment.